In [ ]:
!unzip archive.zip -d /content

Archive:  archive.zip
replace /content/Mental-Health-Twitter.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [ ]:
%%capture
!pip install transformers torch

In [ ]:
%%capture
!pip install "ray[tune]" tune-sklearn
!pip install scikit-optimize
!pip install bayesian-optimization

In [ ]:
%%capture
!pip install accelerate -U

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [ ]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [ ]:
import ray
from ray import tune
from ray.tune.schedulers import HyperBandScheduler
from ray.tune.sklearn import TuneSearchCV
from ray.train import report
from ray import train

In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
from ray.tune.search.bayesopt import BayesOptSearch

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score


In [ ]:
import numpy as np

In [ ]:
# Download necessary NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
df = pd.read_csv('Mental-Health-Twitter.csv')
df.head()

In [ ]:
# Group by user_id and check if all labels are the same for each user
label_consistency = df.groupby('user_id')['label'].nunique()

# Check if all values in label_consistency are 1 (which means each user has only one unique label)
all_same_label = all(label_consistency == 1)

print("If all users have tweets with the same label: ", all_same_label)

In [ ]:
# Function to clean text data
def clean_text(text):
    # Remove retweet headings like "RT @username:"
    text = re.sub(r'RT @\w+:', '', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove special characters and numbers
    text = re.sub(r'\W+|\d+', ' ', text)
    # Lowercase
    text = text.lower()
    # Remove stop words
    stop_words = set(stopwords.words('english'))
    word_tokens = word_tokenize(text)
    filtered_text = [word for word in word_tokens if word not in stop_words]
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    lemmatized_text = [lemmatizer.lemmatize(word) for word in filtered_text]
    return ' '.join(lemmatized_text)

# Apply the cleaning function to post_text column
df['cleaned_post_text'] = df['post_text'].apply(clean_text)

# Display the cleaned text
print(df[['post_text', 'cleaned_post_text']].head())

In [ ]:
from collections import Counter

# Counters for words in tweets with label 0 and label 1
word_counts_0 = Counter()
word_counts_1 = Counter()

for text, label in zip(df['cleaned_post_text'], df['label']):
    tokens = text.split()  # Tokenize the cleaned text
    if label == 0:
        word_counts_0.update(tokens)
    else:
        word_counts_1.update(tokens)


In [ ]:
threshold_ratio = 20  # Set the threshold ratio
significant_words = {}

for word in word_counts_1:
    count_1 = word_counts_1[word]
    count_0 = word_counts_0.get(word, 0)  # Get count from label 0, default to 0 if not found

    # Avoid division by zero; proceed only if the word is present in label 0 tweets
    if count_0 > 0:
        ratio = count_1 / count_0
        if ratio > threshold_ratio:
            significant_words[word] = ratio

# Print the significant words with their ratios
for word, ratio in significant_words.items():
    print(f"Word: '{word}', Ratio: {ratio:.2f}")


Word: 'anxiety', Ratio: 20.20
Word: 'depression', Ratio: 110.75
Word: 'symptom', Ratio: 29.00
Word: 'illness', Ratio: 59.00
Word: 'mental', Ratio: 27.60
Word: 'addiction', Ratio: 82.00
Word: 'suicide', Ratio: 21.67
Word: 'disorder', Ratio: 21.33
Word: 'benefit', Ratio: 29.00
Word: 'treatment', Ratio: 175.00
Word: 'autism', Ratio: 48.00
Word: 'overcome', Ratio: 109.50
Word: 'awareness', Ratio: 27.00
Word: 'motivation', Ratio: 20.50
Word: 'negative', Ratio: 30.50
Word: 'cow', Ratio: 26.00
Word: 'meat', Ratio: 24.00
Word: 'relief', Ratio: 21.00
Word: 'remedy', Ratio: 37.00
Word: 'article', Ratio: 24.50
Word: 'so', Ratio: 96.67
Word: 'leo', Ratio: 30.00
Word: 'jus', Ratio: 21.00


In [ ]:

#Count tweets containing 'depression'
df['contains_depression'] = df['cleaned_post_text'].str.contains('depression', case=False, na=False)
depression_tweets_count = df['contains_depression'].sum()

#Distribution between labels
depression_label_distribution = df[df['contains_depression']].label.value_counts()

# Display the results
print(f"Number of tweets containing 'depression': {depression_tweets_count}")
print("Distribution of 'depression' tweets across labels:")
print(depression_label_distribution)

Number of tweets containing 'depression': 846
Distribution of 'depression' tweets across labels:
1    837
0      9
Name: label, dtype: int64


In [ ]:
import statsmodels.api as sm

# Create a treatment variable
df['treatment'] = df['cleaned_post_text'].str.contains('depression', case=False, na=False).astype(int)

# Define the outcome (label) and treatment variables
X = df[['treatment']]  # Independent variable
y = df['label']  # Dependent variable (assuming 1 for depressed, 0 for not depressed)

# Add a constant to the model (for the intercept)
X = sm.add_constant(X)

# Perform logistic regression
model = sm.Logit(y, X).fit()

# Print the summary of the regression
print(model.summary())


Optimization terminated successfully.
         Current function value: 0.665424
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                  label   No. Observations:                20000
Model:                          Logit   Df Residuals:                    19998
Method:                           MLE   Df Model:                            1
Date:                Thu, 14 Dec 2023   Pseudo R-squ.:                 0.04000
Time:                        05:05:00   Log-Likelihood:                -13308.
converged:                       True   LL-Null:                       -13863.
Covariance Type:            nonrobust   LLR p-value:                3.792e-243
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0865      0.014     -5.981      0.000      -0.115      -0.058
treatment      4.6191      0.

The coefficient of 'treatment' is the log odds of being labeled as depressed when the tweet contains 'depression' vs when it does not. The high coefficient for the treatment variable indicates that the presence of the word 'depression' in a tweet is a strong predictor of the tweet being labeled as depressed in your dataset.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
from torch.utils.data import DataLoader, Dataset
import torch

# Assuming df is your DataFrame and it has columns 'cleaned_post_text' and 'label'
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Create datasets
train_dataset = TextDataset(train_df['cleaned_post_text'].tolist(), train_df['label'].tolist(), tokenizer)
test_dataset = TextDataset(test_df['cleaned_post_text'].tolist(), test_df['label'].tolist(), tokenizer)


In [ ]:
from torch.utils.data import DataLoader

batch_size = 16  # You can modify this based on your GPU capacity

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)


In [ ]:
from transformers import BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup

# Initialize the model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

# Scheduler
epochs = 5
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
def evaluate(model, data_loader, device):
    model_eval = model.eval()

    predictions = []
    true_labels = []

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)

            outputs = model_eval(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            _, preds = torch.max(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    return np.array(predictions), np.array(true_labels)

In [ ]:
import torch.nn as nn

def train_epoch(model, data_loader, optimizer, device, scheduler):
    model = model.train()

    losses = []
    correct_predictions = 0

    for d in data_loader:
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        labels = d["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)
        losses.append(loss.item())

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)

In [ ]:


# Training
for epoch in range(epochs):
    print(f'Epoch {epoch + 1}/{epochs}')
    print('-' * 10)

    train_acc, train_loss = train_epoch(
        model,
        train_dataloader,
        optimizer,
        device,
        scheduler
    )

    print(f'Train loss {train_loss} accuracy {train_acc}')


Epoch 1/5
----------
Train loss 0.4382810356702123 accuracy 0.7846428571428572
Epoch 2/5
----------
Train loss 0.26676242683189255 accuracy 0.8870000000000001
Epoch 3/5
----------
Train loss 0.16435911860609692 accuracy 0.9389285714285714
Epoch 4/5
----------
Train loss 0.10225141789780796 accuracy 0.9696428571428573
Epoch 5/5
----------
Train loss 0.06232148659254225 accuracy 0.9832142857142858


In [ ]:
# Evaluate the model
predictions, true_labels = evaluate(model, test_dataloader, device)

# Calculate Accuracy
accuracy = np.mean(predictions == true_labels)
print(f"Test Accuracy: {accuracy}")

# Classification Report
print("Classification Report")
print(classification_report(true_labels, predictions, target_names=['Class 0', 'Class 1']))

# Confusion Matrix
print("Confusion Matrix")
print(confusion_matrix(true_labels, predictions))


Test Accuracy: 0.8495
Classification Report
              precision    recall  f1-score   support

     Class 0       0.86      0.84      0.85      2983
     Class 1       0.84      0.86      0.85      3017

    accuracy                           0.85      6000
   macro avg       0.85      0.85      0.85      6000
weighted avg       0.85      0.85      0.85      6000

Confusion Matrix
[[2504  479]
 [ 424 2593]]


In [ ]:
# Function to remove 'depression' from a tweet
def remove_depression(text):
    return re.sub(r'\bdepression\b', '', text, flags=re.IGNORECASE)

# Identify tweets containing 'depression' with label 1
depression_tweets = df[(df['cleaned_post_text'].str.contains('depression', case=False, na=False)) & (df['label'] == 1)]

# Randomly select a subset of these tweets to remove 'depression'
np.random.seed(42)  # For reproducibility
remove_indices = np.random.choice(depression_tweets.index, size=int(len(depression_tweets) * 1), replace=False)  # Adjust the percentage as needed

# Remove 'depression' from the selected tweets
df.loc[remove_indices, 'modified_post_text'] = df.loc[remove_indices, 'cleaned_post_text'].apply(remove_depression)
df.loc[~df.index.isin(remove_indices), 'modified_post_text'] = df['cleaned_post_text']


In [ ]:
# Recreate the treatment variable based on the modified text
df['modified_treatment'] = df['modified_post_text'].str.contains('depression', case=False, na=False).astype(int)

# Distribution between labels
print(df.groupby('label')['modified_treatment'].value_counts())

# Logistic Regression for ATE
X_mod = df[['modified_treatment']]
X_mod = sm.add_constant(X_mod)
model_mod = sm.Logit(y, X_mod).fit()
print(model_mod.summary())


label  modified_treatment
0      0                     9991
       1                        9
1      0                     9945
       1                       55
Name: modified_treatment, dtype: int64
Optimization terminated successfully.
         Current function value: 0.692226
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                  label   No. Observations:                20000
Model:                          Logit   Df Residuals:                    19998
Method:                           MLE   Df Model:                            1
Date:                Thu, 14 Dec 2023   Pseudo R-squ.:                0.001329
Time:                        07:36:56   Log-Likelihood:                -13845.
converged:                       True   LL-Null:                       -13863.
Covariance Type:            nonrobust   LLR p-value:                 1.277e-09
                         coef    std err          z      P>|z|   

In [ ]:
df['modified_post_text'].head()

0    year since diagnosed anxiety  today taking mom...
1    sunday need break planning spend little time p...
2                    awake tired need sleep brain idea
3    retro bear make perfect gift great beginner ge...
4    hard say whether packing list making life easi...
Name: modified_post_text, dtype: object

In [ ]:
# Create datasets
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
train_dataset = TextDataset(train_df['modified_post_text'].tolist(), train_df['label'].tolist(), tokenizer)
test_dataset = TextDataset(test_df['modified_post_text'].tolist(), test_df['label'].tolist(), tokenizer)

train_dataloader_no_depression = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader_no_depression = DataLoader(test_dataset, batch_size=batch_size)


In [ ]:
model_no_depression = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model_no_depression.to(device)
# Training
for epoch in range(epochs):
    print(f'Epoch {epoch + 1}/{epochs}')
    print('-' * 10)

    train_acc, train_loss = train_epoch(
        model_no_depression,
        train_dataloader_no_depression,
        optimizer,
        device,
        scheduler
    )

    print(f'Train loss {train_loss} accuracy {train_acc}')


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/5
----------
Train loss 0.7070300827026367 accuracy 0.4882857142857143
Epoch 2/5
----------


KeyboardInterrupt: ignored

In [ ]:
# Evaluate the model
predictions, true_labels = evaluate(model_no_depression, test_dataloader_no_depression, device)

# Calculate Accuracy
accuracy = np.mean(predictions == true_labels)
print(f"Test Accuracy: {accuracy}")

# Classification Report
print("Classification Report")
print(classification_report(true_labels, predictions, target_names=['Class 0', 'Class 1']))

# Confusion Matrix
print("Confusion Matrix")
print(confusion_matrix(true_labels, predictions))

In [ ]:
# Load pre-trained model tokenizer and model
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')

# Function to encode text in batches
def batch_encode(texts, batch_size=32):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded_input = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt')
        with torch.no_grad():
            output = model(**encoded_input)
        embeddings.extend(output.last_hidden_state.detach().cpu().numpy())
    return embeddings

# Apply batch encoding to the cleaned text
df['embeddings'] = batch_encode(df['cleaned_post_text'].tolist())

# Example: Display the embeddings for the first text
print(df['embeddings'].iloc[0])


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

[[-0.17331302 -0.12021988  0.04727072 ... -0.28089187  0.29365286
  -0.00513901]
 [ 0.04024735  0.13801274  0.05358397 ... -0.388307    0.43049568
  -0.40980273]
 [-0.26234692 -0.25665483  0.30996048 ... -0.46868426  0.28122818
   0.14417249]
 ...
 [-0.02850765  0.03714807  0.18374681 ... -0.199296   -0.1318244
  -0.16900803]
 [-0.02219234  0.00570167  0.17162637 ... -0.20851685 -0.03107193
  -0.23720446]
 [-0.09784615 -0.06018201  0.10294885 ... -0.17631516 -0.09361151
  -0.23621081]]


In [ ]:
# Determine the maximum length of the embeddings
max_length = max(len(e) for e in df['embeddings'])

# Function to pad embeddings to the same length
def pad_embedding(embedding):
    # Calculate the number of zeros needed to match the maximum length
    padding_length = max_length - len(embedding)
    # Create a padding array of zeros
    padding = np.zeros((padding_length, embedding.shape[1]))
    # Concatenate the embedding and the padding
    padded_embedding = np.concatenate((embedding, padding))
    return padded_embedding

# Apply padding to each embedding and flatten it
X = np.array([pad_embedding(e).flatten() for e in df['embeddings']])

y = df['label'].values

In [ ]:
# Initialize the model
etc = ExtraTreesClassifier(n_estimators=50)
etc = etc.fit(X, y)

# Select features based on importance
selector = SelectFromModel(etc, prefit=True)

# Transform the dataset
X_new = selector.transform(X)


In [ ]:
X_new.shape

(20000, 13203)

In [ ]:
def train_random_forest(config, X_train, X_test, y_train, y_test):
    # Create and train the model
    model = RandomForestClassifier(
        n_estimators=int(config["n_estimators"]),
        max_depth=int(config["max_depth"]),
        random_state=42
    )
    model.fit(X_train, y_train)

    # Evaluate the model
    accuracy = accuracy_score(y_test, model.predict(X_test))
    train.report({"mean_accuracy": accuracy})


In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=0.3, random_state=42)

# Define the search space
search_space = {
    "n_estimators": tune.uniform(100, 200),
    "max_depth": tune.uniform(20, 40)
}

# Initialize Bayesian optimization search algorithm
bayesopt = BayesOptSearch(metric="mean_accuracy", mode="max")

# Run the experiment
results_bayes = tune.run(
    tune.with_parameters(
        train_random_forest,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test
    ),
    name="bayesian_rf_exp",
    metric="mean_accuracy",
    mode="max",
    stop={"mean_accuracy": 0.99, "training_iteration": 20},
    resources_per_trial={"cpu": 8},  # Adjust based on your system's resources
    config=search_space,
    search_alg=bayesopt,
    num_samples=10
)


2023-12-07 04:22:50,555	INFO tune.py:595 -- [output] This will use the new output engine with verbosity 2. To disable the new output and use the legacy output engine, set the environment variable RAY_AIR_NEW_OUTPUT=0. For more information, please see https://github.com/ray-project/ray/issues/36949


+----------------------------------------------------+
| Configuration for experiment     bayesian_rf_exp   |
+----------------------------------------------------+
| Search algorithm                 SearchGenerator   |
| Scheduler                        FIFOScheduler     |
| Number of trials                 10                |
+----------------------------------------------------+

View detailed results here: /root/ray_results/bayesian_rf_exp
To visualize your results with TensorBoard, run: `tensorboard --logdir /root/ray_results/bayesian_rf_exp`

Trial status: 1 PENDING
Current time: 2023-12-07 04:22:50. Total running time: 0s
Logical resource usage: 0/8 CPUs, 0/0 GPUs
+------------------------------------------------------------------------+
| Trial name                     status       n_estimators     max_depth |
+------------------------------------------------------------------------+
| train_random_forest_4f0c1975   PENDING           195.071       27.4908 |
+-------------------

In [ ]:
# Get the best hyperparameters
best_trial = results_bayes.get_best_trial("mean_accuracy", "max", "all")
best_config = best_trial.config

# Create and train the best model
best_model = RandomForestClassifier(
    n_estimators=int(best_config["n_estimators"]),
    max_depth=int(best_config["max_depth"]),
    random_state=42
)
best_model.fit(X_train, y_train)


RandomForestClassifier(max_depth=26, n_estimators=152, random_state=42)

In [ ]:


# Predict on the test set
y_test_pred = best_model.predict(X_test)

# Calculate accuracy
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy}")

# Additional evaluation metrics
print(classification_report(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))


Test Accuracy: 0.7613333333333333
              precision    recall  f1-score   support

           0       0.75      0.77      0.76      2983
           1       0.77      0.75      0.76      3017

    accuracy                           0.76      6000
   macro avg       0.76      0.76      0.76      6000
weighted avg       0.76      0.76      0.76      6000

[[2305  678]
 [ 754 2263]]
